# Speech-Enhancement — Automated Antigravity Colab Bridge
**Instructions:**
1. Ensure your runtime is set to GPU (`Runtime -> Change runtime type -> T4 GPU`).
2. Click **`Runtime -> Run all`** (or press `Ctrl+F9`).
3. Approve the Google Drive mount prompt when asked.
4. Once Cell 2 prints `BRIDGE ONLINE — Waiting for Antigravity commands...`, the Antigravity agent will automatically run the smoke test and control the workflow!

In [ ]:
# Cell 1: Mount Google Drive, Locate Speech-Enhancement, and Install Dependencies
import os
import glob
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Automatically locate the Speech-Enhancement project folder on Google Drive
candidates = [
    Path('/content/drive/MyDrive/Speech-Enhancement'),
    Path('/content/drive/MyDrive/Desktop/Speech-Enhancement'),
    Path('/content/drive/MyDrive/OneDrive/Desktop/Speech-Enhancement'),
]
PROJECT_DIR = None
for c in candidates:
    if (c / 'configs' / 'config.yaml').exists():
        PROJECT_DIR = c
        break

if PROJECT_DIR is None:
    matches = glob.glob('/content/drive/**/Speech-Enhancement/configs/config.yaml', recursive=True)
    if matches:
        PROJECT_DIR = Path(matches[0]).parent.parent

assert PROJECT_DIR is not None, 'Could not find Speech-Enhancement/configs/config.yaml on Google Drive!'
os.chdir(PROJECT_DIR)
print(f'Active Project Directory: {PROJECT_DIR}')

# Install facebookresearch/denoiser (--no-deps to avoid legacy hydra-core<1.0 conflicts) + requirements
print('Installing dependencies...')
subprocess.run(['pip', 'install', '-q', '--no-deps', 'git+https://github.com/facebookresearch/denoiser.git'], check=True)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Dependencies installed successfully!')

In [ ]:
# Cell 2: Automated Command Bridge Worker Loop (Controlled by Antigravity Agent)
import json
import os
import platform
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
import torch

BRIDGE_DIR = Path(PROJECT_DIR) / 'bridge'
BRIDGE_DIR.mkdir(parents=True, exist_ok=True)

STATUS_FILE = BRIDGE_DIR / 'worker_status.json'
REQ_FILE = BRIDGE_DIR / 'job_request.json'
RES_FILE = BRIDGE_DIR / 'job_result.json'
LOG_FILE = BRIDGE_DIR / 'live_output.log'

def write_json_atomic(path: Path, data: dict):
    tmp = path.with_suffix('.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
status_payload = {
    'state': 'IDLE',
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'python_version': sys.version.split()[0],
    'torch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu_name': gpu_name,
    'project_dir': str(PROJECT_DIR),
    'platform': platform.platform(),
}
write_json_atomic(STATUS_FILE, status_payload)
print('=' * 72)
print(f'BRIDGE ONLINE — GPU: {gpu_name} | PyTorch: {torch.__version__}')
print(f'Watching: {BRIDGE_DIR}')
print('Waiting for Antigravity commands...')
print('=' * 72)

last_job_id = None
if RES_FILE.exists():
    try:
        last_job_id = json.loads(RES_FILE.read_text(encoding='utf-8')).get('job_id')
    except Exception:
        pass

while True:
    try:
        if REQ_FILE.exists():
            req = json.loads(REQ_FILE.read_text(encoding='utf-8'))
            job_id = req.get('job_id')
            cmd = req.get('command')
            action = req.get('action', 'run')

            if action == 'stop_bridge':
                print('Received stop_bridge signal. Exiting loop.')
                break

            if job_id and job_id != last_job_id and cmd:
                print(f'\n[JOB {job_id}] Executing: {cmd}')
                status_payload['state'] = 'RUNNING'
                status_payload['current_job_id'] = job_id
                status_payload['timestamp'] = datetime.now(timezone.utc).isoformat()
                write_json_atomic(STATUS_FILE, status_payload)

                start_t = time.time()
                with open(LOG_FILE, 'w', encoding='utf-8') as log_f:
                    proc = subprocess.Popen(
                        cmd,
                        shell=True,
                        cwd=str(PROJECT_DIR),
                        stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT,
                        text=True,
                        bufsize=1,
                    )
                    output_lines = []
                    for line in proc.stdout:
                        print(line, end='')
                        log_f.write(line)
                        log_f.flush()
                        output_lines.append(line)
                    proc.wait()

                elapsed = round(time.time() - start_t, 2)
                res_payload = {
                    'job_id': job_id,
                    'command': cmd,
                    'returncode': proc.returncode,
                    'elapsed_sec': elapsed,
                    'completed_at': datetime.now(timezone.utc).isoformat(),
                    'output': ''.join(output_lines),
                }
                write_json_atomic(RES_FILE, res_payload)
                last_job_id = job_id

                status_payload['state'] = 'IDLE'
                status_payload['last_completed_job_id'] = job_id
                status_payload['timestamp'] = datetime.now(timezone.utc).isoformat()
                write_json_atomic(STATUS_FILE, status_payload)
                print(f'[JOB {job_id}] Finished with exit code {proc.returncode} in {elapsed}s.')
    except Exception as e:
        print(f'Bridge loop warning: {e}')

    time.sleep(2)